In [23]:
%pip install python-dotenv langchain langchain-classic langchain_core langchain-tavily langchain-community langchain-openai openai langchain-google-genai langchain-anthropic

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

anthropic_key = os.getenv("ANTHROPIC_API_KEY")
print(anthropic_key[:5])

sk-an


In [27]:
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool


class contactInfo(BaseModel):
    name: str
    phone: str
    email: str

anthropic_model = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0
).with_structured_output(contactInfo)
 

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract contact information from the following text."),
    ("user", "{input}")
])  

chain =  prompt | anthropic_model

result = chain.invoke({"input": "What is phone 123-456-7890 of the person named John Doe and email john@example.com?"})

print(result)



name='John Doe' phone='123-456-7890' email='john@example.com'


In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool


class contactInfo(BaseModel):
    name: str
    phone: str
    email: str


anthropic_model = ChatAnthropic(model="claude-sonnet-4-5",
    max_retries=2, # Will wait and try again automatically
    temperature=0) 

# gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
#     max_retries=6, # Will wait and try again automatically
#     temperature=0) 

@tool 
def search(tools: str) -> str:
    """Search for information."""
    return f"Results for: {tools}"

agent = create_agent(model=anthropic_model, 
        tools=[search],
        response_format=ToolStrategy(contactInfo))  
          

result = agent.invoke({"messages": [{"role": "user", 
    "content":"What is phone 123-456-7890 of the person named John Doe and email john@example.com?"}]})


print(result)
print("======")
print(result["structured_response"])



In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool


class contactInfo(BaseModel):
    name: str
    phone: str
    email: str


anthropic_model = ChatAnthropic(model="claude-sonnet-4-5",
    max_retries=2, # Will wait and try again automatically
    temperature=0) 

# gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
#     max_retries=6, # Will wait and try again automatically
#     temperature=0) 

@tool 
def search(tools: str) -> str:
    """Search for information."""
    return "John Doe phone is 123-456-7890 and email is john@example.com"

agent = create_agent(model=anthropic_model, 
        tools=[search],
        response_format=ToolStrategy(contactInfo))  

message = [
            SystemMessage(content="Use the search tool to answer the question."), 
            HumanMessage(content="what is contact information of the person named John Doe?"),
            ToolMessage(content=weather_result,tool_call_id="call_123")  # Must match the call ID
)

        ]  

result = agent.invoke({"messages": message})


print(result)
print("======")
print(result["structured_response"])



{'messages': [SystemMessage(content='Use the search tool to answer the question.', additional_kwargs={}, response_metadata={}, id='9aa2abc4-5e57-4028-987b-79b1ca50f6b7'), HumanMessage(content='what is contact information of the person named John Doe?', additional_kwargs={}, response_metadata={}, id='9ba77a20-041a-4af3-9b53-b575580edbf8'), AIMessage(content=[{'id': 'toolu_01JqHzcpgfY1iSHmjrQ3EnAW', 'input': {'tools': 'John Doe contact information'}, 'name': 'search', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_01TmvwLxpiuTndgVAthoTF28', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 734, 'output_tokens': 40, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_ru